# 棋圣·六道轮回 — 像素画资产 SDXL 重制

**目标**：用 SDXL + Pixel Art XL LoRA + LayerDiffusion（原生透明背景）+ AnimateDiff 重制全部 64 张静态立绘与 10 个动画，替换原 Seedream/rembg 方案。免费跑在 Colab T4 GPU。

**流程**：克隆游戏仓库 → 部署 Forge+扩展+模型 → 启动 Forge API → 生成静态立绘 → 生成动画 → 推回 GitHub。

**运行前**：菜单 `代码执行程序` → `更改运行时类型` → 选 **T4 GPU**。

In [ ]:
# 1. 检查 GPU
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}, CUDA 可用: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无"}')

In [ ]:
# 2. 克隆游戏仓库（获取参考立绘 + 生成脚本）
%cd /content
!rm -rf CHessGAme
!git clone https://github.com/stellarissss/CHessGAme.git
%cd /content/CHessGAme
!git lfs install && git lfs pull 2>/dev/null; echo '仓库就绪'
!ls shared/assets/characters/ | head

In [ ]:
# 3. 克隆 SD WebUI Forge + 扩展（约 2 分钟）
%cd /content
!rm -rf stable-diffusion-webui-forge
!git clone https://github.com/lllyasviel/stable-diffusion-webui-forge.git
%cd /content/stable-diffusion-webui-forge
# LayerDiffusion（原生 RGBA 透明背景）
!git clone https://github.com/lllyasviel/sd-forge-layerdiffuse.git extensions/sd-forge-layerdiffuse
# AnimateDiff（图生视频）
!git clone https://github.com/continue-revolution/sd-webui-animatediff.git extensions/sd-webui-animatediff
print('✅ Forge + 扩展克隆完成')

In [ ]:
# 4. 下载模型（约 10-15 分钟，约 10GB）
import os
root = '/content/stable-diffusion-webui-forge'

# 1) SDXL Base 1.0 (fp16, 6.9GB)
!wget -c -q --show-progress -O {root}/models/Stable-diffusion/sd_xl_base_1.0.safetensors \
  https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors

# 2) SDXL VAE 修复版
!wget -c -q --show-progress -O {root}/models/VAE/sdxl_vae.safetensors \
  https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/diffusion_pytorch_model.safetensors

# 3) Pixel Art XL LoRA（16-bit 像素画风）
!wget -c -q --show-progress -O {root}/models/Lora/pixel-art-xl.safetensors \
  https://huggingface.co/nerijs/pixel-art-xl/resolve/main/pixel-art-xl.safetensors

# 4) LayerDiffusion 透明生成模型（SDXL）—— 扩展会自动下载，但预置更稳
!mkdir -p {root}/models/layer_model
!wget -c -q --show-progress -O {root}/models/layer_model/layer_xl_transparent_attn.safetensors \
  https://huggingface.co/LayerDiffusion/layerdiffusion-v1/resolve/main/layer_xl_transparent_attn.safetensors
!wget -c -q --show-progress -O {root}/models/layer_model/vae_transparent_decoder.safetensors \
  https://huggingface.co/LayerDiffusion/layerdiffusion-v1/resolve/main/vae_transparent_decoder.safetensors

# 5) AnimateDiff SDXL 运动模块 (1.5GB)
!mkdir -p {root}/extensions/sd-webui-animatediff/model
!wget -c -q --show-progress -O {root}/extensions/sd-webui-animatediff/model/mm_sdxl_v10_beta.ckpt \
  https://huggingface.co/guoyww/AnimateDiff/resolve/main/mm_sdxl_v10_beta.ckpt

print('\n✅ 所有模型下载完成')
!du -sh {root}/models/* 2>/dev/null

In [ ]:
# 5. 安装 RIFE（动画插帧 16→48 帧）
%cd /content
!rm -rf RIFE
!git clone https://github.com/hzwer/EasyTemporalKit.git RIFE_repo 2>/dev/null || true
# RIFE 独立版
!rm -rf RIFE
!git clone https://github.com/hzwer/Practical-RIFE.git RIFE 2>/dev/null || git clone https://github.com/hzwer/EasyTemporalKit.git RIFE 2>/dev/null || echo 'RIFE clone 失败，将用复制法兜底'
if os.path.exists('/content/RIFE'):
    %cd /content/RIFE
    !pip install -r requirements.txt -q 2>/dev/null; echo 'RIFE 安装完成'
else:
    print('⚠ RIFE 不可用，动画将用复制法凑帧（不流畅但可用）')

In [ ]:
# 6. 修复依赖 + 后台启动 Forge
# ⚠ Colab 默认 numpy 2.x，与 Forge 的 skimage(np.float_ 已移除)不兼容 → 必须降级
!pip install "numpy<2" -q 2>&1 | tail -3
!python -c "import numpy; print('launch.py 将用 numpy', numpy.__version__, '(应 1.26.x)')"
%cd /content/stable-diffusion-webui-forge
import subprocess, time, os
# 杀掉旧进程
os.system('pkill -f launch.py 2>/dev/null')
time.sleep(2)
forge_proc = subprocess.Popen(
    ['python', 'launch.py', '--api', '--share', '--xformers',
     '--skip-python-version-check',  # 抑制 3.10.6 版本警告
     '--enable-insecure-extension-access', '--no-half-vae', '--theme', 'dark', '--port', '7860'],
    stdout=open('/content/forge.log','w'), stderr=subprocess.STDOUT
)
print(f'Forge PID: {forge_proc.pid}，启动中（日志 /content/forge.log）')
print('等待 Forge 就绪（首次启动会安装依赖，约 5-10 分钟）...')

In [ ]:
# 7. 等待 Forge 就绪并探测 LayerDiffusion
import requests, time
url = 'http://127.0.0.1:7860'
t0 = time.time()
ready = False
while time.time() - t0 < 900:
    try:
        r = requests.get(f'{url}/sdapi/v1/options', timeout=5)
        if r.status_code == 200:
            ready = True; break
    except Exception:
        # 打印最新日志
        try:
            with open('/content/forge.log') as f:
                lines = f.readlines()[-3:]
                print('  '.join(l.strip() for l in lines), flush=True)
        except: pass
    time.sleep(5)
if not ready:
    print('❌ Forge 15分钟未就绪，查看 /content/forge.log'); raise SystemExit
print('✅ Forge 就绪')
# 切换到 SDXL 模型
requests.post(f'{url}/sdapi/v1/options', json={'sd_model_checkpoint': 'sd_xl_base_1.0.safetensors'})
# 探测 LayerDiffusion script
try:
    si = requests.get(f'{url}/sdapi/v1/script-info', timeout=10).json()
    layer = [s for s in si if 'layer' in s.get('name','').lower()]
    print(f'LayerDiffusion scripts: {[s["name"] for s in layer] or "未找到（可能启动后加载）"}')
    print(f'可用 scripts 总数: {len(si)}')
except Exception as e:
    print(f'探测 script-info 失败: {e}')
# 提取 share URL
try:
    with open('/content/forge.log') as f:
        log = f.read()
    import re
    m = re.search(r'https://[\w-]+\.gradio\.live', log)
    if m: print(f'\n🌐 Forge 公网 URL: {m.group()}（可浏览器打开 UI）')
except: pass

## Phase C：生成静态立绘（64 张）
img2img（现有白色背景立绘为参考）+ Pixel Art XL LoRA + LayerDiffusion 透明 → RGBA PNG

In [ ]:
# 8. 生成全部静态立绘（约 60-90 分钟）
%cd /content/CHessGAme/tools
!python forge_generate_portraits.py --skip-if-done
print('\n--- 立绘生成完毕，检查结果 ---')
import glob
from PIL import Image
import numpy as np
# 抽查几张 alpha 质量
for p in sorted(glob.glob('/content/CHessGAme/shared/assets/characters/boy/boy_*.png'))[:3]:
    if '_f' in p: continue
    im = Image.open(p)
    if im.mode=='RGBA':
        a=np.array(im.split()[-1]); print(f'{p.split("/")[-1]}: 不透明={(a>200).mean()*100:.1f}% 半透明={((a>10)&(a<200)).mean()*100:.2f}%')

## Phase D：生成动画（10 个）
AnimateDiff 16帧 → RIFE 插帧 48帧 → LayerDiffusion 透明 → 24FPS 循环

In [ ]:
# 9. 生成全部动画（约 60-120 分钟，每个动画约 6-12 分钟）
%cd /content/CHessGAme/tools
!python forge_generate_animations.py --skip-if-done
print('\n--- 动画生成完毕，检查帧数 ---')
import glob
for char in ['boy','chenmo']:
    for stem in ['determined','happy','neutral','sad','surprised','thinking'] if char=='boy' else ['awkward','smile','surprised','thinking']:
        fs = glob.glob(f'/content/CHessGAme/shared/assets/characters/{char}/{char}_{stem}_f*.png')
        print(f'{char}/{stem}: {len(fs)} 帧')

In [ ]:
# 10. 推回 GitHub（用 gh CLI 或 git）
%cd /content/CHessGAme
# 用 GitHub token 推送（token 从 Colab Secrets 读取，或手动填入）
import os, subprocess
# 方式1: 如已配置 gh
if os.system('gh auth status 2>/dev/null') == 0:
    !git add shared/assets/characters/ && git commit -m "重制像素画立绘与动画: SDXL+PixelArtXL+LayerDiffusion+AnimateDiff" && git push
else:
    print('⚠ gh 未认证。请在下方填入 GitHub Personal Access Token（repo 权限）：')
    print('   取消注释并替换 YOUR_TOKEN')
    # !git remote set-url origin https://YOUR_TOKEN@github.com/stellarissss/CHessGAme.git
    # !git add shared/assets/characters/ && git commit -m "重制像素画立绘与动画" && git push
    print('\n或手动下载：左侧文件浏览器 → CHessGAme/shared/assets/characters/ 下载所需文件')

## 完成

生成完成后，回到 Trae 对话，告诉我已完成。我会从 GitHub 拉取最新资产，集成到前端并验证。

**故障排查**：
- Forge 未就绪：查看 `/content/forge.log`
- LayerDiffusion 未生效：检查 Extensions 页是否启用 sd-forge-layerdiffuse
- AnimateDiff 失败：确认 `mm_sdxl_v10_beta.ckpt` 在 `extensions/sd-webui-animatediff/model/`
- OOM：把 `forge_generate_animations.py` 的 `video_length` 降到 8
- 模型名不对：`GET /sdapi/v1/sd-models` 查看实际模型名